# Preparación de los datos de Alpha Vantage

En este notebook voy a preparar los datos de Alpha Vantage que se utilizarán para entrenar los modelos definitivos. Primero realizaré la limpieza y crearé el objetivo, después calcularé las variables predictoras y finalmente aplicaré la división temporal.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

ruta_raw = ruta_proyecto / "data" / "raw"
ruta_processed = ruta_proyecto / "data" / "processed"

ruta_processed.mkdir(parents=True, exist_ok=True)

archivo_raw = (
    ruta_raw / "eurusd_alpha_vantage_diario.csv"
)

datos = pd.read_csv(
    archivo_raw,
    parse_dates=["Date"],
    index_col="Date"
)

datos = datos.sort_index()

print("Filas cargadas:", len(datos))
print("Primera fecha:", datos.index.min().date())
print("Última fecha:", datos.index.max().date())

display(datos.head())

Filas cargadas: 5000
Primera fecha: 2007-05-22
Última fecha: 2026-07-21


,Open,High,Low,Close
Date,,,,
2007-05-22,1.3467,1.3475,1.3438,1.3447
2007-05-23,1.3448,1.3501,1.3416,1.3460
2007-05-24,1.3458,1.3462,1.3415,1.3428
2007-05-25,1.3428,1.3472,1.3411,1.3442
2007-05-28,1.3449,1.3461,1.3445,1.3451


In [2]:
columnas_precios = [
    "Open",
    "High",
    "Low",
    "Close"
]

# Trabajo sobre una copia de los datos originales
datos_limpios = datos[columnas_precios].copy()

# Elimino duplicados y filas sin precios completos
datos_limpios = datos_limpios[
    ~datos_limpios.index.duplicated(keep="first")
]

datos_limpios = datos_limpios.dropna(
    subset=columnas_precios
)

datos_limpios = datos_limpios.sort_index()

print("Filas después de la limpieza:", len(datos_limpios))
print("Fechas duplicadas:", datos_limpios.index.duplicated().sum())
print("Valores faltantes:", datos_limpios.isna().sum().sum())

Filas después de la limpieza: 5000
Fechas duplicadas: 0
Valores faltantes: 0


In [3]:
# Calculo cuánto cambió el cierre respecto a la jornada anterior
datos_limpios["Retorno_diario"] = (
    datos_limpios["Close"].pct_change()
)

# Comparo el cierre siguiente con el cierre actual
cierre_siguiente = datos_limpios["Close"].shift(-1)

datos_limpios["Objetivo"] = (
    cierre_siguiente > datos_limpios["Close"]
).astype("Int64")

# La última jornada queda vacía porque todavía no conozco su resultado
datos_limpios.loc[
    cierre_siguiente.isna(),
    "Objetivo"
] = pd.NA

display(
    datos_limpios[
        ["Close", "Retorno_diario", "Objetivo"]
    ].tail()
)

,Close,Retorno_diario,Objetivo
Date,,,
2026-07-15,1.1463,0.003853,0
2026-07-16,1.1441,-0.001919,0
2026-07-17,1.1439,-0.000175,0
2026-07-20,1.1414,-0.002186,0
2026-07-21,1.1397,-0.001489,<NA>


In [4]:
archivo_limpio = (
    ruta_processed
    / "eurusd_alpha_vantage_limpio.csv"
)

datos_limpios.to_csv(
    archivo_limpio,
    index=True,
    encoding="utf-8"
)

print("Archivo guardado en:")
print(archivo_limpio)

print("\nDimensiones:", datos_limpios.shape)
print(
    "Filas con objetivo conocido:",
    datos_limpios["Objetivo"].notna().sum()
)
print(
    "Filas para predicción futura:",
    datos_limpios["Objetivo"].isna().sum()
)

Archivo guardado en:
C:\TFM_EURUSD\data\processed\eurusd_alpha_vantage_limpio.csv

Dimensiones: (5000, 6)
Filas con objetivo conocido: 4999
Filas para predicción futura: 1


## Creación de variables predictoras

A partir de los datos limpios voy a calcular las mismas variables utilizadas en el análisis inicial. Estas variables representan movimientos anteriores, características de la vela, tendencia, volatilidad e impulso.

In [5]:
# Trabajo sobre una copia de los datos limpios
datos_modelo = datos_limpios.copy()

# Guardo los retornos de jornadas anteriores
for rezago in [1, 2, 3, 5]:
    datos_modelo[f"Retorno_lag_{rezago}"] = (
        datos_modelo["Retorno_diario"].shift(rezago)
    )

# Calculo cuánto se movió el precio durante la jornada
datos_modelo["Rango_diario"] = (
    (datos_modelo["High"] - datos_modelo["Low"])
    / datos_modelo["Open"]
)

# Calculo si el cierre quedó arriba o abajo de la apertura
datos_modelo["Cuerpo_vela"] = (
    (datos_modelo["Close"] - datos_modelo["Open"])
    / datos_modelo["Open"]
)

# Calculo en qué parte del rango diario quedó el cierre
rango = datos_modelo["High"] - datos_modelo["Low"]

datos_modelo["Posicion_cierre"] = np.where(
    rango != 0,
    (datos_modelo["Close"] - datos_modelo["Low"]) / rango,
    0.5
)

# Calculo la distancia del cierre respecto a las medias móviles
for ventana in [5, 10, 20]:
    media_movil = (
        datos_modelo["Close"]
        .rolling(window=ventana)
        .mean()
    )

    datos_modelo[f"Distancia_MA{ventana}"] = (
        datos_modelo["Close"] / media_movil - 1
    )

# Calculo la volatilidad de los retornos recientes
for ventana in [5, 20]:
    datos_modelo[f"Volatilidad_{ventana}"] = (
        datos_modelo["Retorno_diario"]
        .rolling(window=ventana)
        .std()
    )

# Calculo el RSI de 14 jornadas
cambio = datos_modelo["Close"].diff()

ganancias = cambio.clip(lower=0)
perdidas = -cambio.clip(upper=0)

media_ganancias = (
    ganancias
    .rolling(window=14)
    .mean()
)

media_perdidas = (
    perdidas
    .rolling(window=14)
    .mean()
)

rs = media_ganancias / media_perdidas

datos_modelo["RSI_14"] = (
    100 - (100 / (1 + rs))
)

# Si no hubo ningún movimiento durante las 14 jornadas, dejo el RSI neutral
sin_movimiento = (
    (media_ganancias == 0)
    & (media_perdidas == 0)
)

datos_modelo.loc[
    sin_movimiento,
    "RSI_14"
] = 50

# Calculo el histograma del MACD
ema_12 = (
    datos_modelo["Close"]
    .ewm(span=12, adjust=False)
    .mean()
)

ema_26 = (
    datos_modelo["Close"]
    .ewm(span=26, adjust=False)
    .mean()
)

macd = ema_12 - ema_26

senal_macd = (
    macd
    .ewm(span=9, adjust=False)
    .mean()
)

datos_modelo["MACD_hist"] = (
    macd - senal_macd
)

In [6]:
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

print(
    "Número de variables predictoras:",
    len(variables_predictoras)
)

print("\nVariables creadas:")

for variable in variables_predictoras:
    print("-", variable)

Número de variables predictoras: 15

Variables creadas:
- Retorno_diario
- Retorno_lag_1
- Retorno_lag_2
- Retorno_lag_3
- Retorno_lag_5
- Rango_diario
- Cuerpo_vela
- Posicion_cierre
- Distancia_MA5
- Distancia_MA10
- Distancia_MA20
- Volatilidad_5
- Volatilidad_20
- RSI_14
- MACD_hist


In [7]:
# Reviso los valores faltantes generados por los cálculos iniciales
faltantes = (
    datos_modelo[variables_predictoras]
    .isna()
    .sum()
)

print("Valores faltantes antes de limpiar:")
print(faltantes[faltantes > 0])

filas_antes = len(datos_modelo)

# Elimino las primeras jornadas que todavía no tienen suficiente historial
datos_modelo = datos_modelo.dropna(
    subset=variables_predictoras
).copy()

filas_despues = len(datos_modelo)

columnas_base = [
    "Open",
    "High",
    "Low",
    "Close"
]

columnas_finales = (
    columnas_base
    + variables_predictoras
    + ["Objetivo"]
)

datos_modelo = datos_modelo[columnas_finales]

# Compruebo que todas las variables sean válidas
if datos_modelo[variables_predictoras].isna().any().any():
    raise ValueError(
        "Todavía existen valores faltantes en las variables predictoras."
    )

valores_infinitos = np.isinf(
    datos_modelo[variables_predictoras].to_numpy()
).sum()

if valores_infinitos > 0:
    raise ValueError(
        "Existen valores infinitos en las variables predictoras."
    )

print("\nFilas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

print("\nDimensiones finales:", datos_modelo.shape)

print(
    "Filas con objetivo conocido:",
    datos_modelo["Objetivo"].notna().sum()
)

print(
    "Filas para predicción futura:",
    datos_modelo["Objetivo"].isna().sum()
)

display(datos_modelo.tail())

Valores faltantes antes de limpiar:
Retorno_diario     1
Retorno_lag_1      2
Retorno_lag_2      3
Retorno_lag_3      4
Retorno_lag_5      6
Distancia_MA5      4
Distancia_MA10     9
Distancia_MA20    19
Volatilidad_5      5
Volatilidad_20    20
RSI_14            14
dtype: int64

Filas antes: 5000
Filas después: 4980
Filas eliminadas: 20

Dimensiones finales: (4980, 20)
Filas con objetivo conocido: 4979
Filas para predicción futura: 1


,Open,High,Low,Close,Retorno_diario,Retorno_lag_1,Retorno_lag_2,Retorno_lag_3,Retorno_lag_5,Rango_diario,Cuerpo_vela,Posicion_cierre,Distancia_MA5,Distancia_MA10,Distancia_MA20,Volatilidad_5,Volatilidad_20,RSI_14,MACD_hist,Objetivo
Date,,,,,,,,,,,,,,,,,,,,
2026-07-15,1.1419,1.1482,1.1405,1.1463,0.003853,0.003339,-0.002804,-0.001313,0.000263,0.006743,0.003853,0.753247,0.003695,0.003467,0.004240,0.002889,0.002746,63.988095,0.001099,0
2026-07-16,1.1464,1.1476,1.1430,1.1441,-0.001919,0.003853,0.003339,-0.002804,0.001227,0.004013,-0.002006,0.239130,0.001541,0.001444,0.002379,0.003122,0.002649,58.430233,0.001164,0
2026-07-17,1.1441,1.1452,1.1424,1.1439,-0.000175,-0.001919,0.003853,0.003339,-0.001313,0.002447,-0.000175,0.535714,0.000910,0.001234,0.002331,0.003022,0.002636,53.074434,0.001167,0
2026-07-20,1.1432,1.1449,1.1402,1.1414,-0.002186,-0.000175,-0.001919,0.003853,-0.002804,0.004111,-0.001575,0.255319,-0.001854,-0.000727,0.000197,0.002863,0.002558,48.948949,0.000981,0
2026-07-21,1.1413,1.1428,1.1396,1.1397,-0.001489,-0.002186,-0.000175,-0.001919,0.003339,0.002804,-0.001402,0.031250,-0.002957,-0.002093,-0.001363,0.002491,0.002409,53.442623,0.000736,<NA>


In [8]:
archivo_variables = (
    ruta_processed
    / "eurusd_alpha_vantage_variables.csv"
)

datos_modelo.to_csv(
    archivo_variables,
    index=True,
    encoding="utf-8"
)

print("Dataset guardado en:")
print(archivo_variables)

Dataset guardado en:
C:\TFM_EURUSD\data\processed\eurusd_alpha_vantage_variables.csv


## División temporal

Los datos se dividen respetando el orden cronológico. La partición se asigna según la fecha de la jornada que se quiere predecir, para evitar que entrenamiento o validación utilicen resultados pertenecientes al periodo siguiente.

- Entrenamiento: objetivo hasta el 31/12/2022
- Validación: objetivo entre 2023 y 2024
- Prueba final: objetivo desde 2025
- Predicción futura: última jornada sin objetivo conocido

In [9]:
FECHA_FIN_ENTRENAMIENTO = pd.Timestamp("2022-12-31")
FECHA_FIN_VALIDACION = pd.Timestamp("2024-12-31")

# Trabajo sobre una copia del dataset con las variables
datos_particionados = datos_modelo.copy()

# Cada fila intenta predecir el resultado de la jornada siguiente
fecha_objetivo = (
    datos_particionados.index
    .to_series()
    .shift(-1)
)

objetivo_conocido = (
    datos_particionados["Objetivo"].notna()
)

mascara_entrenamiento = (
    objetivo_conocido
    & (fecha_objetivo <= FECHA_FIN_ENTRENAMIENTO)
)

mascara_validacion = (
    objetivo_conocido
    & (fecha_objetivo > FECHA_FIN_ENTRENAMIENTO)
    & (fecha_objetivo <= FECHA_FIN_VALIDACION)
)

mascara_prueba = (
    objetivo_conocido
    & (fecha_objetivo > FECHA_FIN_VALIDACION)
)

mascara_futuro = (
    datos_particionados["Objetivo"].isna()
)

datos_particionados["Particion"] = ""

datos_particionados.loc[
    mascara_entrenamiento,
    "Particion"
] = "entrenamiento"

datos_particionados.loc[
    mascara_validacion,
    "Particion"
] = "validacion"

datos_particionados.loc[
    mascara_prueba,
    "Particion"
] = "prueba"

datos_particionados.loc[
    mascara_futuro,
    "Particion"
] = "futuro"

In [10]:
# Compruebo que todas las filas estén dentro de una partición
if (datos_particionados["Particion"] == "").any():
    raise ValueError(
        "Existen filas que no fueron asignadas a ninguna partición."
    )

# Compruebo que cada resultado pertenezca al periodo correcto
if not (
    fecha_objetivo[mascara_entrenamiento]
    <= FECHA_FIN_ENTRENAMIENTO
).all():
    raise ValueError(
        "Entrenamiento contiene resultados posteriores a 2022."
    )

if not (
    (
        fecha_objetivo[mascara_validacion]
        > FECHA_FIN_ENTRENAMIENTO
    )
    & (
        fecha_objetivo[mascara_validacion]
        <= FECHA_FIN_VALIDACION
    )
).all():
    raise ValueError(
        "Validación contiene resultados fuera de 2023 y 2024."
    )

if not (
    fecha_objetivo[mascara_prueba]
    > FECHA_FIN_VALIDACION
).all():
    raise ValueError(
        "Prueba contiene resultados anteriores a 2025."
    )

print("División temporal realizada correctamente")

División temporal realizada correctamente


In [11]:
entrenamiento = datos_particionados[
    datos_particionados["Particion"] == "entrenamiento"
]

validacion = datos_particionados[
    datos_particionados["Particion"] == "validacion"
]

prueba = datos_particionados[
    datos_particionados["Particion"] == "prueba"
]

futuro = datos_particionados[
    datos_particionados["Particion"] == "futuro"
]

resumen_particiones = pd.DataFrame({
    "Filas": [
        len(entrenamiento),
        len(validacion),
        len(prueba),
        len(futuro)
    ],
    "Primera fecha disponible": [
        entrenamiento.index.min().date(),
        validacion.index.min().date(),
        prueba.index.min().date(),
        futuro.index.min().date()
    ],
    "Última fecha disponible": [
        entrenamiento.index.max().date(),
        validacion.index.max().date(),
        prueba.index.max().date(),
        futuro.index.max().date()
    ]
}, index=[
    "Entrenamiento",
    "Validación",
    "Prueba final",
    "Predicción futura"
])

display(resumen_particiones)

,Filas,Primera fecha disponible,Última fecha disponible
Entrenamiento,4052,2007-06-19,2022-12-29
Validación,522,2022-12-30,2024-12-30
Prueba final,405,2024-12-31,2026-07-20
Predicción futura,1,2026-07-21,2026-07-21


In [12]:
archivo_particiones = (
    ruta_processed
    / "eurusd_alpha_vantage_particiones.csv"
)

datos_particionados.to_csv(
    archivo_particiones,
    index=True,
    encoding="utf-8"
)

print("Dataset guardado en:")
print(archivo_particiones)

print(
    "\nTotal de filas:",
    len(datos_particionados)
)

print(
    "Filas sin partición:",
    (datos_particionados["Particion"] == "").sum()
)

Dataset guardado en:
C:\TFM_EURUSD\data\processed\eurusd_alpha_vantage_particiones.csv

Total de filas: 4980
Filas sin partición: 0
